In [2]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [6]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

display(run(load_queries("sql/01_scope.sql"), "scope_reconcile"))

,raw_rows,scoped_rows,rows_removed_by_scope
0,20405666,20405666,0


In [7]:
waits = load_queries("sql/01_audit_waits.sql")

for name in waits:
    print(f"\n=== {name} ===")
    display(run(waits, name))


=== wait_percentiles_valid ===


,hvfhs_license_num,n_valid,mean_min,p50_min,p75_min,p90_min,p90_seconds,p95_min,p99_min,p999_min,max_min
0,HV0003,15235488,4.556,3.800,5.517,8.033,482.0,10.217,15.783,25.467,178.350
1,HV0005,4970390,4.691,4.083,5.767,8.033,482.0,9.867,14.533,23.217,115.583



=== wait_bins_valid ===


,hvfhs_license_num,wait_bin,n_rows,pct_of_company_valid
0,HV0003,00: exactly 0 s,169,0.001
1,HV0003,01: under 1 min,208487,1.368
2,HV0003,02: 1 to 2 min,1546422,10.150
3,HV0003,03: 2 to 5 min,8785859,57.667
4,HV0003,04: 5 to 10 min,3879439,25.463
5,HV0003,05: 10 to 20 min,765899,5.027
6,HV0003,06: 20 to 30 min,41566,0.273
7,HV0003,07: 30 to 60 min,6965,0.046
8,HV0003,08: 1 to 2 h,659,0.004
9,HV0003,09: 2 h or more,23,0.000



=== longest_valid_waits ===


,hvfhs_license_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,wait_min,PULocationID,DOLocationID,trip_miles,trip_time,driver_pay
0,HV0003,2025-01-19 18:55:00,2025-01-19 18:56:05,2025-01-19 21:53:21,2025-01-19 23:07:58,178.4,132,265,43.38,4477,137.00
1,HV0003,2025-01-19 21:53:00,2025-01-19 21:40:44,2025-01-20 00:26:46,2025-01-20 01:31:14,153.8,132,265,42.76,3868,131.77
2,HV0003,2025-01-13 20:38:00,2025-01-13 20:28:05,2025-01-13 23:08:42,2025-01-14 00:00:35,150.7,132,230,19.05,3113,91.71
3,HV0003,2025-01-19 20:30:00,2025-01-19 20:07:10,2025-01-19 22:59:26,2025-01-19 23:49:48,149.4,132,265,22.30,3022,88.58
4,HV0003,2025-01-20 05:52:00,2025-01-20 05:37:40,2025-01-20 08:16:25,2025-01-20 10:19:11,144.4,132,265,124.14,7366,298.82
5,HV0003,2025-01-14 11:43:00,2025-01-14 11:37:56,2025-01-14 14:05:12,2025-01-14 16:11:29,142.2,132,265,94.56,7577,246.05
6,HV0003,2025-01-17 18:09:49,2025-01-17 20:31:29,2025-01-17 20:31:29,2025-01-17 20:34:31,141.7,230,161,0.37,182,6.78
7,HV0003,2025-01-16 10:10:00,2025-01-16 10:10:04,2025-01-16 12:31:16,2025-01-16 13:01:02,141.3,132,265,16.65,1786,65.48
8,HV0003,2025-01-29 14:48:19,2025-01-29 17:08:01,2025-01-29 17:08:01,2025-01-29 19:11:11,139.7,89,89,0.05,7390,2.68
9,HV0003,2025-01-14 18:57:00,2025-01-14 18:31:22,2025-01-14 21:15:25,2025-01-14 22:32:27,138.4,132,265,53.95,4622,129.83


... (4 outputs left out)


=== negative_airport_profile ===


,hvfhs_license_num,wait_group,n_rows,pct_dropoff_airport_zone,pct_pickup_airport_zone
0,HV0003,negative_wait,120967,32.84,0.60
1,HV0003,nonneg_wait,15235488,3.98,3.71
2,HV0005,negative_wait,78821,25.29,0.39
3,HV0005,nonneg_wait,4970390,4.08,3.26



=== uber_b_sensitivity ===


,n_valid_uber,n_pattern_b,median_min_as_is,median_min_without_b,median_min_b_as_high,p90_min_as_is,p90_min_without_b,p90_min_b_as_high
0,15235488,58321,3.8,3.8,3.817,8.033,8.033,8.15


In [8]:
money = load_queries("sql/01_audit_money.sql")
cnt = run(money, "nonpositive_counts")
display(cnt)
rates = cnt.set_index("hvfhs_license_num")
display((rates.drop(columns="n_rows").div(rates["n_rows"], axis=0) * 100).round(3))  # % of company rows
for name in money:
    if name != "nonpositive_counts":
        print(f"\n=== {name} ===")
        display(run(money, name))

,hvfhs_license_num,n_rows,miles_zero,time_zero,fare_negative,fare_zero,pay_negative,pay_zero,pay_gt_fare
0,HV0003,15356455,2398,4,2393,832,313,4225,2358327
1,HV0005,5049211,208,0,34,13945,0,0,318745


,miles_zero,time_zero,fare_negative,fare_zero,pay_negative,pay_zero,pay_gt_fare
hvfhs_license_num,,,,,,,
HV0003,0.016,0.0,0.016,0.005,0.002,0.028,15.357
HV0005,0.004,0.0,0.001,0.276,0.000,0.000,6.313



=== negative_money_profile ===


,fare_sign,pay_sign,n_rows,miles_zero,time_zero,other_money_negative,median_miles,median_trip_min,min_fare,min_pay
0,fare_negative,pay_positive,2427,0,0,0,1.850,10.75,-27.98,2.01
1,fare_positive,pay_negative,313,0,0,0,24.820,54.13,3.58,-55.02
2,fare_positive,pay_positive,20383924,2126,3,0,2.830,15.05,0.01,0.07
3,fare_positive,pay_zero,4225,480,1,0,0.320,3.68,0.51,0.00
4,fare_zero,pay_positive,14777,0,0,0,2.989,14.18,0.00,2.00



=== pay_to_fare_ratio ===


,hvfhs_license_num,n_rows_used,n_pay_gt_fare,pct_pay_gt_fare,p01,p25,p50,p75,p99
0,HV0003,15352917,2355102,15.340,0.449,0.597,0.733,0.910,1.567
1,HV0005,5035232,304766,6.053,0.446,0.662,0.749,0.844,1.233



=== pay_ratio_tails ===


,hvfhs_license_num,n_rows_used,pay_per_mile_q,pay_per_mile_max,pay_per_min_q,pay_per_min_max
0,HV0003,15353744,"[1.327095683572827, 2.2099065118894856, 4.5962...",7110.0,"[0.32857142857142857, 0.528293185403507, 0.928...",167.15
1,HV0005,5049003,"[2.225651593865345, 2.40596468033049, 4.730538...",5470.0,"[0.6498069527655735, 0.6888268156424582, 0.892...",1378.20



=== implied_speed_buckets ===


,hvfhs_license_num,mph_bucket,n_rows,pct_of_company
0,HV0003,0: zero mph (zero miles),2394,0.016
1,HV0003,1: under 1 mph,2499,0.016
2,HV0003,2: 1 to 5 mph,294085,1.915
3,HV0003,3: 5 to 15 mph,10053803,65.470
4,HV0003,4: 15 to 30 mph,4264134,27.768
5,HV0003,5: 30 to 60 mph,739362,4.815
6,HV0003,6: 60 to 100 mph,173,0.001
7,HV0003,7: 100 mph or more,1,0.000
8,HV0005,0: zero mph (zero miles),208,0.004
9,HV0005,1: under 1 mph,269,0.005
